# Task 3 — Multi-agent financial research (LangGraph)
Produces `task3_agentic/logs/agent_trace.jsonl` and `task3_agentic/memory/<TICKER>_<date>.json`.

In [1]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("../..")
from dotenv import load_dotenv; load_dotenv()
import pandas as pd
from task3_agentic.src import tracing, memory
from task3_agentic.src.tools import get_price_data, get_news, calculate_volatility, llm_sentiment, web_search, call_tool
from task3_agentic.src.graph import run_single_agent, ask, run_multi_agent
from task3_agentic.src.pipeline import print_messages, print_report

TICKER = "AAPL"
tracing.TRACE_PATH.unlink(missing_ok=True)          # fresh trace for this notebook run
for f in memory.MEMORY_DIR.glob(f"{TICKER}_*.json"): f.unlink()
print("GROQ key set:", bool(os.environ.get("GROQ_API_KEY")))

GROQ key set: True


## 3A — The five tools (each returns a JSON dict with `ok`)

In [2]:
tracing.set_agent("tool_check")
checks = {
    "get_price_data": call_tool(get_price_data, ticker=TICKER, period="2y"),
    "get_news": call_tool(get_news, ticker=TICKER, n=10),
    "calculate_volatility": call_tool(calculate_volatility, ticker=TICKER, window=30),
    "web_search": call_tool(web_search, query=f"{TICKER} analyst outlook"),
}
checks["llm_sentiment"] = call_tool(llm_sentiment, headlines=checks["get_news"]["headlines"])
for name, out in checks.items():
    print(f"{name:22s} ok={out['ok']} type={type(out).__name__} -> {json.dumps(out, default=str)[:160]}")
print("\nHeadlines:", len(checks["get_news"]["headlines"]))
print(pd.DataFrame(checks["llm_sentiment"].get("items", []))[["headline", "sentiment", "confidence", "brief_reason"]].to_string())

get_price_data         ok=True type=dict -> {"ok": true, "ticker": "AAPL", "as_of": "2026-10-12", "period": "2y", "rows": 500, "current_price": 340.9700012207031, "high_52w": 345.3399963378906, "low_52w":
get_news               ok=True type=dict -> {"ok": true, "ticker": "AAPL", "count": 10, "headlines": ["Apple Prepares New Smart Home Devices Featuring AI-Enhanced Siri", "Apple Needs Phones and Smart Home
calculate_volatility   ok=True type=dict -> {"ok": true, "ticker": "AAPL", "window": 30, "annualized_volatility": 0.2131187750711461, "observations": 30}
web_search             ok=True type=dict -> {"ok": true, "query": "AAPL analyst outlook", "results": [{"title": "Apple (AAPL) Stock Forecast & Analyst Price Targets", "snippet": "Stock forecasts and analy
llm_sentiment          ok=True type=dict -> {"ok": true, "items": [{"headline": "Apple Prepares New Smart Home Devices Featuring AI-Enhanced Siri", "sentiment": "positive", "confidence": 0.85, "brief_reas

Headlines: 10
           

## 3A — Autonomous ReAct agent
The tool order is not scripted. To show the observe-then-replan cycle, the first `get_news` call is forced to return an empty list. The agent observes that and switches to `web_search`.

In [3]:
agent, messages = run_single_agent(TICKER, demo_empty_news=True)
print_messages(messages)

[HumanMessage] Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

[AIMessage] TOOL CALL get_price_data({'period': '1y', 'ticker': 'AAPL'})
[ToolMessage] {"ok": true, "ticker": "AAPL", "as_of": "2026-10-12", "period": "1y", "rows": 251, "current_price": 340.9700012207031, "high_52w": 345.3399963378906, "low_52w": 243.4199981689453, "sma_50": 323.0298010253906, "sma_200": 291.086499786377, "rsi_14": 60.2067386766452, "macd": 3.637335951062198, "macd_s

[AIMessage] TOOL CALL get_news({'n': 10, 'ticker': 'AAPL'})
[ToolMessage] {"ok": true, "ticker": "AAPL", "source": "forced_empty", "count": 0, "headlines": []}

[AIMessage] TOOL CALL web_search({'query': 'Apple Inc AAPL news October 2026'})
[ToolMessage] {"ok": true, "query": "Apple Inc AAPL news October 2026", "results": [{"title": "Bullish or Bearish on AAPL? - Leveraged & Inverse AAPL ETFs", "snippet": "Sing

## 3C — Short-term memory: follow-up answered from the conversation, no tool re-call

In [4]:
before_lines, before_msgs = tracing.count_trace_lines(), len(messages)
follow = ask(agent, "What annualized volatility did you compute, and over what window?", thread_id=f"{TICKER}-session")
new = follow[before_msgs:]
print_messages(new)
print("tool calls in follow-up:", sum(len(getattr(m, 'tool_calls', []) or []) for m in new),
      "| trace lines before/after:", before_lines, tracing.count_trace_lines())

[HumanMessage] What annualized volatility did you compute, and over what window?

[AIMessage] The **annualised historical volatility** I calculated for Apple (AAPL) is **21.3 %** (0.2131 in decimal form).  

- **Look‑back window:** 30 trading days (the default window used by the `calculate_volatility` tool).  

This figure reflects the standard deviation of daily log returns over the most recent 30‑day period, annualised to a yearly basis.

tool calls in follow-up: 0 | trace lines before/after: 11 11


## 3B — Two agents with restricted tools, a structured handoff, and a critique loop
- **Agent A, data analyst:** `get_price_data`, `calculate_volatility`, `llm_sentiment`.
- **Agent B, research writer:** `get_news`, `web_search`.

A hands off a Pydantic `DataBrief`. B requests 90-day volatility, A answers with a `ClarificationResponse`, and B writes the report.

In [5]:
tracing.set_agent("research_writer")
result = run_multi_agent(TICKER, use_cache=False)
print("cache_hit:", result["cache_hit"], "| report source:", result.get("report_source"), "\n")
print("\n".join(result["steps"]))
print()
print_report(result["final_report"])

cache_hit: False | report source: llm 

[writer] get_news -> ok=True count=10
[analyst] price ok, vol(30)=21.31%, sentiment=positive
[analyst -> writer] DataBrief handed off:
{
  "ticker": "AAPL",
  "as_of": "2026-10-12",
  "price": {
    "ticker": "AAPL",
    "as_of": "2026-10-12",
    "period": "2y",
    "rows": 500,
    "current_price": 340.9700012207031,
    "high_52w": 345.3399963378906,
    "low_52w": 243.4199981689453,
    "sma_50": 323.0298010253906,
    "sma_200": 291.086499786377,
    "rsi_14": 60.20673859193633,
    "macd": 3.6373359508481826,
    "macd_signal": 3.8235248117284844,
    "macd_histogram": -0.18618886088030173,
    "bollinger_upper": 342.71980947072797,
    "bollinger_middle": 335.7835037231445,
    "bollinger_lower": 328.8471979755611
  },
  "volatility": {
    "ticker": "AAPL",
    "window": 30,
    "annualized_volatility": 0.2131187750711461,
    "observations": 30
  },
  "sentiment": {
    "overall_sentiment": "positive",
    "overall_score": 0.4345
  },
  

## 3C — Persistent memory: second run loads the cached brief

In [6]:
before = tracing.count_trace_lines()
again = run_multi_agent(TICKER, use_cache=True)
print("cache file:", memory.brief_path(TICKER))
print("cache_hit:", again["cache_hit"], "| new tool calls:", tracing.count_trace_lines() - before)
print("same report:", again["final_report"] == result["final_report"])

cache file: /Users/saluka/Repos/ML_assignment/task3_agentic/memory/AAPL_2026-10-13.json
cache_hit: True | new tool calls: 0
same report: True


## Observability — agent_trace.jsonl

In [7]:
trace = pd.read_json(tracing.TRACE_PATH, lines=True)
print(len(trace), "tool calls logged")
trace[["agent", "tool", "inputs", "duration_ms", "output"]]

17 tool calls logged


,agent,tool,inputs,duration_ms,output
0,tool_check,get_price_data,"{'ticker': 'AAPL', 'period': '2y'}",890.41,"{""ok"": true, ""ticker"": ""AAPL"", ""as_of"": ""2026-..."
1,tool_check,get_news,"{'ticker': 'AAPL', 'n': 10}",814.67,"{""ok"": true, ""ticker"": ""AAPL"", ""count"": 10, ""h..."
2,tool_check,calculate_volatility,"{'ticker': 'AAPL', 'window': 30}",131.12,"{""ok"": true, ""ticker"": ""AAPL"", ""window"": 30, ""..."
3,tool_check,web_search,{'query': 'AAPL analyst outlook'},4949.41,"{""ok"": true, ""query"": ""AAPL analyst outlook"", ..."
4,tool_check,llm_sentiment,{'headlines': ['Apple Prepares New Smart Home ...,2625.96,"{""ok"": true, ""items"": [{""headline"": ""Apple Pre..."
5,single_agent,get_price_data,"{'ticker': 'AAPL', 'period': '1y'}",246.79,"{""ok"": true, ""ticker"": ""AAPL"", ""as_of"": ""2026-..."
6,single_agent,get_news,"{'ticker': 'AAPL', 'n': 10}",0.00,"{""ok"": true, ""ticker"": ""AAPL"", ""source"": ""forc..."
7,single_agent,web_search,{'query': 'Apple Inc AAPL news October 2026'},2484.17,"{""ok"": true, ""query"": ""Apple Inc AAPL news Oct..."
8,single_agent,llm_sentiment,{'headlines': ['Bullish or Bearish on AAPL? - ...,1317.74,"{""ok"": true, ""items"": [{""headline"": ""Bullish o..."
9,single_agent,calculate_volatility,"{'ticker': 'AAPL', 'window': 30}",294.72,"{""ok"": true, ""ticker"": ""AAPL"", ""window"": 30, ""..."
